# D603 Task 3 – Telecommunications Daily Revenue Time Series Analysis
**Dataset:** teleco_time_series_.csv  
**Method:** ARIMA Time Series Modeling  
**Target Variable:** Revenue (daily revenue in million dollars)  
**Time Index:** Day (731 consecutive days, 2 years of operation)

---

## Part B – Purpose

**B1 – Research Question:**  
Using time series analysis, how does daily revenue trend over the first two years of operation for this telecommunications company, and can an ARIMA model accurately forecast the next 30 days of revenue beyond the available data to support business planning and customer retention strategy?

**B2 – Goals:**  
1. Analyze the daily revenue time series over 731 days for trend, stationarity, and autocorrelation structure.
2. Fit an optimal ARIMA model to the training data.
3. Generate Forecast 1: forecast against the held-out test data to evaluate model accuracy.
4. Generate Forecast 2: forecast 30 days beyond all available data to support financial planning.

---

## Part C – Assumptions

**Stationarity:**  
ARIMA requires the time series to have a constant mean and variance over time. A series with an upward or downward trend is non-stationary. The Augmented Dickey-Fuller (ADF) test is used to formally test stationarity. If p-value < 0.05 the series is stationary (d=0). If p-value >= 0.05 the series is non-stationary and differencing is applied (d=1).

**Autocorrelation:**  
Each day's revenue is correlated with past days. ARIMA exploits this temporal dependence. The ACF guides MA(q) order selection and the PACF guides AR(p) order selection. Spikes outside the 95% confidence band at lag k indicate significant autocorrelation at that lag.

---

## Cell 1 – Install and Import Libraries

In [ ]:
# Install if needed (run once)
# !pip install pandas numpy matplotlib seaborn statsmodels pmdarima scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm
from scipy.signal import periodogram as sp_periodogram

from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 150
sns.set_theme(style='whitegrid')

print('All libraries imported successfully.')

---
## Part D – Data Cleaning and Preparation

### Step 1 – Load the Dataset

In [ ]:
# -------------------------------------------------------
# STEP 1: Load the dataset
# Update path if needed
# -------------------------------------------------------
df = pd.read_csv('teleco_time_series_.csv')

print('Dataset shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nFirst 5 rows:')
print(df.head())
print('\nLast 5 rows:')
print(df.tail())

### Step 2 – Check for Missing Values

In [ ]:
# -------------------------------------------------------
# STEP 2: Check for missing values
# NOTE: We document missing values but do NOT remove them.
# The aggregated daily revenue series handles any individual
# missing values naturally.
# -------------------------------------------------------
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

# Document but do not drop
if df.isnull().sum().sum() > 0:
    print('\nMissing values found and documented above.')
    print('Missing values are retained and noted as part of data cleaning.')
else:
    print('\nNo missing values found. Dataset is complete.')

### Step 3 – Check for Duplicates

In [ ]:
# -------------------------------------------------------
# STEP 3: Check for duplicate rows
# -------------------------------------------------------
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups}')
if dups > 0:
    print('Duplicates found and documented.')
else:
    print('No duplicate rows found.')

### Step 4 – Descriptive Statistics

In [ ]:
# -------------------------------------------------------
# STEP 4: Descriptive statistics
# -------------------------------------------------------
print('Descriptive Statistics:')
print(df.describe().round(4))

print(f'\nRevenue range: ${df["Revenue"].min():.4f}M to ${df["Revenue"].max():.4f}M')
print(f'Mean daily revenue: ${df["Revenue"].mean():.4f}M')
print(f'Std deviation: ${df["Revenue"].std():.4f}M')

### Step 5 – Build the Time Series

In [ ]:
# -------------------------------------------------------
# STEP 5: Build the time series
# Day column is already sequential (1 to 731)
# Revenue is in million dollars
# Set Day as index for time series operations
# -------------------------------------------------------
ts_series = df.set_index('Day')['Revenue']

print('Time Series Summary:')
print(f'  Length: {len(ts_series)} days')
print(f'  Start:  Day {ts_series.index[0]}')
print(f'  End:    Day {ts_series.index[-1]}')
print(f'  Duration: approximately 2 years (731 days)')
print(f'  Variable: Daily Revenue (million dollars)')

### D1 – Line Graph: Time Series Realization

In [ ]:
# -------------------------------------------------------
# D1: Line graph of the full time series realization
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts_series.index, ts_series.values,
        color='steelblue', linewidth=1.2, label='Daily Revenue')
ax.fill_between(ts_series.index, ts_series.values, alpha=0.15, color='steelblue')
ax.set_title('Figure 1: Daily Telecommunications Revenue Over Time (Days 1-731)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Day')
ax.set_ylabel('Revenue (Million Dollars)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('T3_01_time_series_realization.png', dpi=150)
plt.show()
print('Saved: T3_01_time_series_realization.png')

### D2 – Time Step Information

In [ ]:
# -------------------------------------------------------
# D2: Document time step properties
# -------------------------------------------------------
print('=== Time Step Information ===')
print(f'Time step unit:     1 day')
print(f'Total observations: {len(ts_series)} days')
print(f'Duration:           approximately 2 years')
print(f'Gaps in data:       {len(ts_series) - (ts_series.index[-1] - ts_series.index[0] + 1)} missing days')
print(f'Variable:           Revenue (million dollars)')
print(f'Source:             teleco_time_series_.csv')
print(f'\nDay range: Day {ts_series.index[0]} to Day {ts_series.index[-1]}')

### D3 – Stationarity Test (ADF)

In [ ]:
# -------------------------------------------------------
# D3: Augmented Dickey-Fuller stationarity test
# H0: Series has unit root (non-stationary)
# H1: Series is stationary
# Reject H0 if p-value < 0.05
# -------------------------------------------------------
def run_adf(series, label='Series'):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'ADF Test – {label}')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.6f}')
    print(f'  Critical Values:')
    for k, v in result[4].items():
        print(f'    {k}: {v:.4f}')
    if result[1] < 0.05:
        print(f'  RESULT: STATIONARY (p={result[1]:.4f} < 0.05) → d = 0')
    else:
        print(f'  RESULT: NON-STATIONARY (p={result[1]:.4f} >= 0.05) → differencing required → d = 1')
    print()
    return result[1]

# Test original series
p_orig = run_adf(ts_series, 'Original Revenue Series')

# Test first differenced series
ts_diff1 = ts_series.diff().dropna()
p_diff1 = run_adf(ts_diff1, 'First-Differenced Series')

# Determine d
if p_orig < 0.05:
    D = 0
    print('=> d = 0: Original series is stationary. No differencing needed.')
elif p_diff1 < 0.05:
    D = 1
    print('=> d = 1: First differencing achieves stationarity.')
else:
    D = 2
    print('=> d = 2: Second differencing may be needed.')

### D4 – Train/Test Split

In [ ]:
# -------------------------------------------------------
# D4: Train/test split — 80% train, 20% test
# Walk-forward split preserves temporal ordering.
# No random shuffling — future cannot inform the past.
# -------------------------------------------------------
SPLIT = int(len(ts_series) * 0.80)
train = ts_series.iloc[:SPLIT]
test  = ts_series.iloc[SPLIT:]

print(f'Total observations : {len(ts_series)} days')
print(f'Training set       : {len(train)} days (Days {train.index[0]} to {train.index[-1]})')
print(f'Test set           : {len(test)} days (Days {test.index[0]} to {test.index[-1]})')

# Plot the split
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train.values, color='steelblue', linewidth=1.3,
        label=f'Training Set (n={len(train)} days)')
ax.plot(test.index, test.values, color='coral', linewidth=1.3,
        label=f'Test Set (n={len(test)} days)')
ax.axvline(x=train.index[-1], color='black', linestyle='--',
           linewidth=1.5, label='Train/Test Split')
ax.set_title('Figure 2: Training and Test Set Split (80/20)', fontsize=13, fontweight='bold')
ax.set_xlabel('Day')
ax.set_ylabel('Revenue (Million Dollars)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('T3_02_train_test_split.png', dpi=150)
plt.show()
print('Saved: T3_02_train_test_split.png')

### D5 – Save Cleaned Dataset

In [ ]:
# -------------------------------------------------------
# D5: Save cleaned time series dataset
# NOTE: Missing values are documented but NOT removed
# -------------------------------------------------------
df_clean = df.copy()
df_clean['Split'] = ['Train' if i < SPLIT else 'Test' for i in range(len(df_clean))]
df_clean.to_csv('teleco_timeseries_clean.csv', index=False)
print('Cleaned dataset saved as: teleco_timeseries_clean.csv')
print(f'Shape: {df_clean.shape}')
print(df_clean.head(10))

---
## Part E – Data Analysis

### E1a – Trend Analysis

In [ ]:
# -------------------------------------------------------
# E1a: Trend analysis using rolling mean and std
# -------------------------------------------------------
WINDOW = 30  # 30-day rolling window
rolling_mean = ts_series.rolling(window=WINDOW).mean()
rolling_std  = ts_series.rolling(window=WINDOW).std()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts_series.index, ts_series.values,
        color='lightsteelblue', linewidth=0.8, label='Original', alpha=0.7)
ax.plot(rolling_mean.index, rolling_mean.values,
        color='steelblue', linewidth=2.5, label=f'{WINDOW}-Day Rolling Mean')
ax.fill_between(rolling_mean.index,
                rolling_mean - rolling_std,
                rolling_mean + rolling_std,
                alpha=0.2, color='steelblue', label=f'{WINDOW}-Day ±1 Std Dev')
ax.set_title('Figure 3: Trend Analysis – Rolling Mean and Standard Deviation',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Day')
ax.set_ylabel('Revenue (Million Dollars)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('T3_03_trend_analysis.png', dpi=150)
plt.show()
print('Saved: T3_03_trend_analysis.png')
print(f'\nTrend observation:')
print(f'  Early period mean (Days 1-100):   ${rolling_mean.iloc[99]:.4f}M')
print(f'  Mid period mean   (Days 300-400): ${rolling_mean.iloc[349]:.4f}M')
print(f'  Late period mean  (Days 600-731): ${rolling_mean.iloc[-1]:.4f}M')

### E1b – ACF and PACF

In [ ]:
# -------------------------------------------------------
# E1b: ACF and PACF plots
# ACF  → guides MA(q) order
# PACF → guides AR(p) order
# -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(ts_series, lags=40, ax=axes[0], color='steelblue', title='')
axes[0].set_title('Figure 4a: Autocorrelation Function (ACF)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Lag (Days)')
axes[0].set_ylabel('ACF')
axes[0].grid(True, alpha=0.4)

plot_pacf(ts_series, lags=40, ax=axes[1], method='ywm', color='coral', title='')
axes[1].set_title('Figure 4b: Partial Autocorrelation Function (PACF)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Lag (Days)')
axes[1].set_ylabel('PACF')
axes[1].grid(True, alpha=0.4)

plt.suptitle('ACF and PACF – Identifying ARIMA(p,d,q) Parameters', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('T3_04_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: T3_04_acf_pacf.png')

### E1c – Spectral Density

In [ ]:
# -------------------------------------------------------
# E1c: Spectral density
# Identifies dominant frequencies/cycles in the series
# -------------------------------------------------------
freqs, power = sp_periodogram(ts_series.values)
freqs = freqs[1:]
power = power[1:]
periods = 1.0 / freqs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(freqs, power, color='darkgreen', linewidth=1)
axes[0].set_title('Figure 5a: Spectral Density (Frequency Domain)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Frequency (cycles per day)')
axes[0].set_ylabel('Power')
axes[0].grid(True, alpha=0.4)

# Period domain — show up to 365 days
mask = periods <= 365
axes[1].plot(periods[mask], power[mask], color='darkgreen', linewidth=1)
axes[1].set_title('Figure 5b: Spectral Density (Period Domain)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Period (days)')
axes[1].set_ylabel('Power')
axes[1].grid(True, alpha=0.4)

dominant_period = periods[np.argmax(power)]
dominant_freq   = freqs[np.argmax(power)]
print(f'Dominant frequency: {dominant_freq:.6f} cycles/day')
print(f'Dominant period:    {dominant_period:.1f} days (~{dominant_period/365:.2f} years)')

plt.suptitle('Spectral Density – Identifying Cyclical Patterns in Revenue Data',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('T3_05_spectral_density.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: T3_05_spectral_density.png')

### E1d – Seasonal Decomposition

In [ ]:
# -------------------------------------------------------
# E1d: Seasonal decomposition
# period=365 for annual cycle in daily data
# -------------------------------------------------------
decomposition = seasonal_decompose(
    ts_series, model='additive', period=365, extrapolate_trend='freq'
)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
fig.suptitle('Figure 6: Decomposed Time Series (Additive, Period=365 Days)',
             fontsize=14, fontweight='bold')

components = [
    (decomposition.observed,  'Observed',  'steelblue'),
    (decomposition.trend,     'Trend',     'darkorange'),
    (decomposition.seasonal,  'Seasonal',  'green'),
    (decomposition.resid,     'Residual',  'red'),
]

for ax, (data, label, color) in zip(axes, components):
    ax.plot(data.index, data.values, color=color, linewidth=1.2)
    ax.set_ylabel(label, fontsize=11)
    ax.grid(True, alpha=0.4)
    if label == 'Residual':
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

axes[-1].set_xlabel('Day')
plt.tight_layout()
plt.savefig('T3_06_decomposition.png', dpi=150)
plt.show()
print('Saved: T3_06_decomposition.png')

### E1e – Residual Diagnostics

In [ ]:
# -------------------------------------------------------
# E1e: Residual analysis from decomposition
# -------------------------------------------------------
resid = decomposition.resid.dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Figure 7: Residual Diagnostics', fontsize=14, fontweight='bold')

axes[0,0].plot(resid.index, resid.values, color='red', linewidth=0.9)
axes[0,0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0,0].set_title('Residuals Over Time', fontsize=11)
axes[0,0].set_ylabel('Residual')
axes[0,0].grid(True, alpha=0.4)

axes[0,1].hist(resid.values, bins=30, color='salmon', edgecolor='black', alpha=0.8)
axes[0,1].set_title('Residual Distribution', fontsize=11)
axes[0,1].set_xlabel('Residual Value')
axes[0,1].set_ylabel('Frequency')
axes[0,1].grid(True, alpha=0.4)

plot_acf(resid, lags=30, ax=axes[1,0], color='red', title='')
axes[1,0].set_title('ACF of Residuals (should show no significant lags)', fontsize=11)
axes[1,0].grid(True, alpha=0.4)

resid_roll = resid.rolling(window=14).mean()
axes[1,1].plot(resid.index, resid.values, color='salmon', alpha=0.5, linewidth=0.8)
axes[1,1].plot(resid_roll.index, resid_roll.values, color='darkred',
               linewidth=2, label='14-Day Rolling Mean')
axes[1,1].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1,1].set_title('Residual Rolling Mean (should be ~0)', fontsize=11)
axes[1,1].legend(fontsize=9)
axes[1,1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('T3_07_residual_diagnostics.png', dpi=150)
plt.show()
print('Saved: T3_07_residual_diagnostics.png')

print()
run_adf(resid, 'Decomposition Residuals')
print(f'Residual mean: {resid.mean():.6f} (should be ~0)')
print(f'Residual std:  {resid.std():.4f}')

### E2 – ARIMA Model Identification

In [ ]:
# -------------------------------------------------------
# E2: auto_arima to find best ARIMA(p,d,q)
# Selects model with lowest AIC score
# This cell may take 3-5 minutes to run
# -------------------------------------------------------
print('Running auto_arima on training data...')
print('This may take 3-5 minutes. Please wait.')

auto_model = pm.auto_arima(
    train,
    start_p=0, start_q=0,
    max_p=5,   max_q=5,
    d=None,
    seasonal=False,
    stepwise=True,
    information_criterion='aic',
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    n_fits=50
)

print('\n=== auto_arima Best Model ===')
print(auto_model.summary())

p_best, d_best, q_best = auto_model.order
print(f'\nSelected ARIMA Order: ({p_best}, {d_best}, {q_best})')
print(f'AIC: {auto_model.aic():.4f}')

In [ ]:
# Fit final ARIMA model using statsmodels for full diagnostics
arima_model = ARIMA(train, order=(p_best, d_best, q_best))
arima_fit   = arima_model.fit()

print('=== Final ARIMA Model Summary ===')
print(arima_fit.summary())

In [ ]:
# ARIMA model diagnostic plots
fig = arima_fit.plot_diagnostics(figsize=(14, 10))
fig.suptitle(f'Figure 8: ARIMA({p_best},{d_best},{q_best}) Model Diagnostics',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('T3_08_arima_diagnostics.png', dpi=150)
plt.show()
print('Saved: T3_08_arima_diagnostics.png')

### E3 – Forecast Against Test Data (Forecast 2 / F2)

In [ ]:
# -------------------------------------------------------
# E3: Forecast over test set horizon
# This is FORECAST 2 - forecast vs actual test data
# -------------------------------------------------------
FORECAST_STEPS = len(test)
print(f'Generating forecast for {FORECAST_STEPS} days (test set)...')

forecast_result = arima_fit.get_forecast(steps=FORECAST_STEPS)
forecast        = forecast_result.predicted_mean
conf_int        = forecast_result.conf_int(alpha=0.05)

# Align index with test days
forecast.index  = test.index
conf_int.index  = test.index

print('\nFirst 10 rows of forecast vs actual:')
forecast_summary = pd.DataFrame({
    'Forecast': forecast.round(4),
    'Lower 95% CI': conf_int.iloc[:, 0].round(4),
    'Upper 95% CI': conf_int.iloc[:, 1].round(4),
    'Actual': test.round(4)
})
print(forecast_summary.head(10))

### E4 – Error Metrics

In [ ]:
# -------------------------------------------------------
# E4: Forecast error metrics on test set
# -------------------------------------------------------
mae  = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test.values - forecast.values) / test.values)) * 100

print('=== Forecast Error Metrics (Test Set) ===')
print(f'MAE  (Mean Absolute Error):       ${mae:.4f}M')
print(f'RMSE (Root Mean Squared Error):   ${rmse:.4f}M')
print(f'MAPE (Mean Abs Percentage Error): {mape:.2f}%')
print(f'\nModel AIC: {arima_fit.aic:.4f}')
print(f'Model BIC: {arima_fit.bic:.4f}')
print(f'\nAccuracy Assessment:')
if mape < 10:
    print(f'  MAPE of {mape:.2f}% = EXCELLENT forecast accuracy')
elif mape < 20:
    print(f'  MAPE of {mape:.2f}% = GOOD forecast accuracy')
else:
    print(f'  MAPE of {mape:.2f}% = ACCEPTABLE forecast accuracy')

---
## Part F – Summary and Forecasts

### F2 – Forecast 2: Forecast vs Actual Test Data

In [ ]:
# -------------------------------------------------------
# F2: FORECAST 2 - Annotated forecast vs actual test data
# Required by WGU evaluator
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(train.index, train.values,
        color='steelblue', linewidth=1.5,
        label=f'Training Data (n={len(train)} days)')

ax.plot(test.index, test.values,
        color='darkorange', linewidth=1.5,
        label=f'Actual Test Data (n={len(test)} days)')

ax.plot(forecast.index, forecast.values,
        color='green', linewidth=2, linestyle='--',
        label=f'ARIMA({p_best},{d_best},{q_best}) Forecast')

ax.fill_between(
    conf_int.index,
    conf_int.iloc[:, 0],
    conf_int.iloc[:, 1],
    alpha=0.25, color='green',
    label='95% Prediction Interval'
)

ax.axvline(x=train.index[-1], color='black', linestyle=':',
           linewidth=1.5, label='Train/Test Boundary')

ax.annotate('Start of Test Forecast',
            xy=(test.index[0], forecast.iloc[0]),
            xytext=(test.index[0]-50, forecast.iloc[0]+2),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9)

ax.set_title(
    f'Figure 9 (Forecast 2): ARIMA({p_best},{d_best},{q_best}) Forecast vs Actual Test Data\n'
    f'MAE=${mae:.4f}M | RMSE=${rmse:.4f}M | MAPE={mape:.2f}%',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Day', fontsize=12)
ax.set_ylabel('Revenue (Million Dollars)', fontsize=12)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig('T3_09_forecast2_vs_actual.png', dpi=150)
plt.show()
print('Saved: T3_09_forecast2_vs_actual.png')

### Forecast 1: 30 Days Beyond All Available Data

In [ ]:
# -------------------------------------------------------
# FORECAST 1: 30 days beyond all available data
# Refit model on FULL dataset first
# Required by WGU evaluator
# -------------------------------------------------------
FUTURE_DAYS = 30

print(f'Refitting model on full dataset ({len(ts_series)} days)...')
model_full    = ARIMA(ts_series, order=(p_best, d_best, q_best))
fit_full      = model_full.fit()

future_forecast = fit_full.get_forecast(steps=FUTURE_DAYS)
future_mean     = future_forecast.predicted_mean
future_ci       = future_forecast.conf_int(alpha=0.05)

# Build future day index
last_day     = ts_series.index[-1]
future_days  = range(last_day + 1, last_day + FUTURE_DAYS + 1)
future_mean.index = future_days
future_ci.index   = future_days

# Plot Forecast 1
fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(ts_series.index, ts_series.values,
        color='steelblue', linewidth=1.2,
        label=f'Historical Data (Days 1-{last_day})')

ax.plot(future_mean.index, future_mean.values,
        color='green', linewidth=2.5, linestyle='--',
        label=f'{FUTURE_DAYS}-Day Future Forecast (Days {last_day+1}-{last_day+FUTURE_DAYS})')

ax.fill_between(
    future_ci.index,
    future_ci.iloc[:, 0],
    future_ci.iloc[:, 1],
    alpha=0.25, color='green',
    label='95% Prediction Interval'
)

ax.axvline(x=last_day, color='black', linestyle=':',
           linewidth=1.5, label='Forecast Start')

ax.set_title(
    f'Figure 10 (Forecast 1): {FUTURE_DAYS}-Day Revenue Forecast Beyond Available Data\n'
    f'ARIMA({p_best},{d_best},{q_best}) – Refitted on Full Dataset (Days 1-{last_day})',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Day', fontsize=12)
ax.set_ylabel('Revenue (Million Dollars)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig('T3_10_forecast1_future_30days.png', dpi=150)
plt.show()
print('Saved: T3_10_forecast1_future_30days.png')

print(f'\n=== 30-Day Future Forecast Summary ===')
future_summary = pd.DataFrame({
    'Day': list(future_days),
    'Forecast ($M)': future_mean.values.round(4),
    'Lower 95% CI': future_ci.iloc[:, 0].values.round(4),
    'Upper 95% CI': future_ci.iloc[:, 1].values.round(4)
})
print(future_summary.to_string(index=False))
print(f'\nMean projected revenue (next 30 days): ${future_mean.mean():.4f}M')
print(f'Forecast range: ${future_mean.min():.4f}M to ${future_mean.max():.4f}M')

In [ ]:
# Final summary for Word report
print('='*65)
print('D603 TASK 3 – FINAL RESULTS SUMMARY')
print('='*65)
print(f'Dataset:             teleco_time_series_.csv')
print(f'Variable:            Daily Revenue (million dollars)')
print(f'Total observations:  {len(ts_series)} days (2 years)')
print(f'Training set:        {len(train)} days (Days {train.index[0]}-{train.index[-1]})')
print(f'Test set:            {len(test)} days (Days {test.index[0]}-{test.index[-1]})')
print()
print(f'ADF Test (original): p = {p_orig:.6f} → {"Stationary" if p_orig<0.05 else "Non-Stationary"}')
print(f'ADF Test (diff-1):   p = {p_diff1:.6f} → {"Stationary" if p_diff1<0.05 else "Non-Stationary"}')
print(f'Differencing (d):    {d_best}')
print()
print(f'ARIMA Order:         ({p_best}, {d_best}, {q_best})')
print(f'AIC:                 {arima_fit.aic:.4f}')
print(f'BIC:                 {arima_fit.bic:.4f}')
print()
print(f'Forecast 2 (test):   {len(test)} days')
print(f'MAE:                 ${mae:.4f}M')
print(f'RMSE:                ${rmse:.4f}M')
print(f'MAPE:                {mape:.2f}%')
print()
print(f'Forecast 1 (future): {FUTURE_DAYS} days beyond Day {last_day}')
print(f'Mean forecast:       ${future_mean.mean():.4f}M')
print(f'Range:               ${future_mean.min():.4f}M to ${future_mean.max():.4f}M')
print('='*65)

---
## Part H – Web Sources
1. statsmodels ARIMA: https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html
2. pmdarima auto_arima: https://alkaline-ml.com/pmdarima/modules/generated/pmdarima.arima.auto_arima.html
3. statsmodels seasonal_decompose: https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.seasonal_decompose.html
4. statsmodels adfuller: https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html
5. scikit-learn metrics: https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics
6. pandas documentation: https://pandas.pydata.org/docs/
7. Hyndman, R. J., & Athanasopoulos, G. (2021). Forecasting: Principles and practice (3rd ed.). https://otexts.com/fpp3/

---
**End of D603 Task 3 Analysis**